In [1]:
import json
import os
import chromadb
from typing import Annotated, TYPE_CHECKING

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent,FunctionResultContent, StreamingTextContent, ChatMessageContent
from semantic_kernel.contents.utils.author_role import AuthorRole
from semantic_kernel.functions import kernel_function

from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.contents import AuthorRole

if TYPE_CHECKING:
    from chromadb.api.models.Collection import Collection
# Initialize the asynchronous OpenAI client
from dotenv import load_dotenv


In [2]:
from typing import Annotated, Dict, Any, List
from semantic_kernel.functions import kernel_function

In [ ]:
import json
import os

# ------------------------------
# 动态加载 subcommand 信息
# ------------------------------
def load_subcommands(info_path="../json/modkit_subcommand_info.json", param_path="../json/modkit_subcommand_parameter.json"):
    with open(info_path, "r") as f:
        subcommand_info = json.load(f)
    with open(param_path, "r") as f:
        subcommand_parameter = json.load(f)
    return subcommand_info, subcommand_parameter

# 使用示例
SUBCOMMANDINFO, subcommand_parameter = load_subcommands()

In [4]:
import json
from typing import Dict, Any

class CodeGeneratorPlugin:
    def __init__(self, subcommand_parameter_dict: Dict[str, Any] = None):
        self.subcommand_parameter = subcommand_parameter_dict

    @kernel_function(
        description="Get the subcommand parameter",
        name="get_subcommand_parameter"
    )
    def get_subcommand_parameter(self, subcommand_name: str) -> Dict[str, Any]:
        return self.subcommand_parameter[subcommand_name]

    @kernel_function(
        description="Generate a prompt for the LLM given tool parameters JSON and the user's task request.",
        name="generate_tool_prompt"
    )
    def generate_tool_prompt(self,tool_definition: Dict[str, Any], user_task: str) -> str:
        """
        Generate a prompt for the LLM given tool parameters JSON and the user's task request.
        English only.
        """
        formatted_params = json.dumps(tool_definition, indent=2, ensure_ascii=False)
        prompt = (
            "You are an expert systems engineer skilled in integrating command-line tools. "
            "You will be given a parameter definition and a user task. "
            "Your goal is to generate accurate codes that fulfills the user's request.\n\n"
            f"User Task:\n{user_task}\n\n"
            "Parameter Definition:\n"
            f"{formatted_params}\n\n"
            "Instructions:\n"
            "1. Use relevant optional parameters if they match the user's intent.\n"
            "2. Only use the parameters that are shown in the parameter definition. Never use parameters that are not in the parameter definition."
            "3. Do not use space as a parameter value. Use a name instead."
        )
        return prompt
    


In [5]:
# ------------------------------
# 1️⃣ 初始化 OpenAI 客户端
# ------------------------------
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com/"
)

chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

In [ ]:
from pydantic import BaseModel, ValidationError, Field

class SubTask(BaseModel):
    assigned_subcommand: str = Field(
        description="The specific subcommand assigned to handle this subtask")
    task_details: str = Field(
        description="Detailed description of what needs to be done for this subtask")


class ModkitPlan(BaseModel):
    main_task: str = Field(
        description="The overall travel request from the user")
    subtasks: List[SubTask] = Field(
        description="List of subtasks broken down from the main task, each assigned to a specialized subcommand")

In [7]:
from semantic_kernel.functions import KernelArguments
AGENT_NAME = "ModkitAgent"

AGENT_INSTRUCTIONS = """You are an planner agent.
    Your job is to decide which subcommand to run based on the user's request.
    Below are the available agents specialised in different tasks:
"""

for name, info in SUBCOMMANDINFO.items():
    AGENT_INSTRUCTIONS += ("\n - "+name+": "+info.get("description")+"\t"+ "Input: "+info.get("input"))


# Create the prompt execution settings and configure the Pydantic model response format
settings = OpenAIChatPromptExecutionSettings(response_format=ModkitPlan)

agent = ChatCompletionAgent(
    service=chat_completion_service,
    description="You are an planner agent.",
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    arguments=KernelArguments(settings) 
)

In [8]:
from semantic_kernel.functions import KernelArguments
AGENT_NAME = "CodeGeneratorAgent"

AGENT_INSTRUCTIONS = """You are an code generator agent.
    Your job is to generate the code based on the user's request and the subcommand parameter description.
    You should follow the steps:
    - Generate the code prompt based on the user's request and the subcommand parameter description.
    - Generate the code based on the code prompt in step 1.
    - Explain the parameter in the code.

    Important:
    - Your code section should start with <code> and end with </code>.
    - Your code MUST start with 'modkit '
"""

code_agent = ChatCompletionAgent(
    service=chat_completion_service,
    description="You are an code generator agent.",
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    plugins=[CodeGeneratorPlugin(subcommand_parameter)],
)

In [9]:
from semantic_kernel.agents import SequentialOrchestration, GroupChatOrchestration, RoundRobinGroupChatManager

from semantic_kernel.contents import ChatMessageContent

def agent_response_callback(message: ChatMessageContent) -> None:
    print(f"# {message.name}\n{message.content}")


In [56]:
from semantic_kernel.contents import ChatHistorySummarizationReducer

# Configure reduction parameters
REDUCER_TARGET_COUNT = 1  # Target number of messages to keep after reduction
REDUCER_THRESHOLD =  4 # Trigger reduction when message count exceeds this

history_reducer = ChatHistorySummarizationReducer(
    service=chat_completion_service,
    target_count=REDUCER_TARGET_COUNT,
    threshold_count=REDUCER_THRESHOLD,
)

chat = SequentialOrchestration(
    members=[agent, code_agent],
    agent_response_callback=agent_response_callback,
)

# GroupChatOrchestration(
#     members=[agent, code_agent],
#     manager=RoundRobinGroupChatManager(max_rounds=5),  # Odd number so writer gets the last word
#     agent_response_callback=agent_response_callback,
# )

In [ ]:
from semantic_kernel.agents.runtime import InProcessRuntime
runtime = InProcessRuntime()
runtime.start()

# ModkitAgent
{"main_task":"Retrieve read-level base modification information from the provided BAM file.","subtasks":[{"assigned_subcommand":"extract","task_details":"Extract read-level base modification information from the modBAM file and produce a TSV with positions, probabilities, and modification context."}]}
# CodeGeneratorAgent

# CodeGeneratorAgent

# CodeGeneratorAgent

# CodeGeneratorAgent
### Code Prompt

Generate code to extract read-level base modification information from a modBAM file into a TSV format. The code should utilize the following parameters for the extraction process:
- `IN_BAM`: Input modBAM file with MM and ML tags containing base modification information (required).
- `OUT_TSV`: Output TSV table of per-read modification calls and probabilities (required).
- `--ref`: Reference FASTA file for coordinate alignment and motif labeling (optional).
- `--include-mods`: Comma-separated list of modification codes to extract (optional).
- `--combine-strands`: Combine

In [ ]:
user_inputs = ["I want to know each site modification information in the bam file.", "My bam file is mod.sorted.bam","my reference genome is /athena/chenlab/scratch/ziw4007/llm/ONT_plus/ref/ref.fa"]

async def main():
    thread = ChatHistoryAgentThread(chat_history=history_reducer)
    for user_input in user_inputs:
        history_reducer.add_user_message(user_input)
        orchestration_result = await chat.invoke(
            task=history_reducer.messages,
            runtime=runtime,
        )
        value = await orchestration_result.get(timeout=100)
        print(f"***** Final Result *****\n{value}")
        history_reducer.add_assistant_message(value.content)

        if len(thread) > 4:
            await thread.reduce()
    await runtime.stop_when_idle()

await main()

***** Final Result *****
### Code Prompt

Generate code to extract read-level base modification information from a modBAM file into a TSV format. The code should utilize the following parameters for the extraction process:
- `IN_BAM`: Input modBAM file with MM and ML tags containing base modification information (required).
- `OUT_TSV`: Output TSV table of per-read modification calls and probabilities (required).
- `--ref`: Reference FASTA file for coordinate alignment and motif labeling (optional).
- `--include-mods`: Comma-separated list of modification codes to extract (optional).
- `--combine-strands`: Combine calls across strands (output strand column as '.') (optional).
- `--min-prob`: Minimum probability threshold for including modification calls (default: 0.5) (optional).
- `--threads`: Number of threads for parallel extraction (default: 4) (optional).
- `--log-filepath`: Write logs to this file (optional).
- `--log-level`: Set logging verbosity (default: INFO, choices: DEBUG, 

In [65]:
# history_reducer.messages[0].content
# history_reducer.messages[1].content
history_reducer.messages[2].content


"### Code Prompt\nI need to extract read-level base modification information from a modBAM file named 'mod.sorted.bam' using the reference genome located at '/athena/chenlab/scratch/ziw4007/llm/ONT_plus/ref/ref.fa'. The output should be a TSV file with positions, probabilities, and modification context.\n\n### Generated Code\n<code>\nmodkit extract --ref '/athena/chenlab/scratch/ziw4007/llm/ONT_plus/ref/ref.fa' \\\n               --min-prob 0.5 \\\n               --threads 4 \\\n               --output-column-descriptions \\\n               'mod.sorted.bam' 'output.tsv'\n</code>\n\n### Explanation of the Parameters\n- **`--ref`**: This specifies the reference FASTA file used for coordinate alignment and motif labeling.\n- **`--min-prob`**: This sets the minimum probability threshold (default is 0.5) for including modification calls in the output.\n- **`--threads`**: This specifies the number of threads (default is 4) to utilize for parallel extraction, enabling faster processing.\n- **

In [28]:
turn_scratchpad_ops = 0
async for msg in thread.get_messages():
    if hasattr(msg, 'content') and msg.content:
        content_str = str(msg.content)
        if 'read_scratchpad' in content_str or 'update_scratchpad' in content_str:
            turn_scratchpad_ops += 1

In [ ]:
# Cell 13: Interactive chat and plan display (revised as per notebook's style)
from IPython.display import display, HTML

async def interactive_chat():
    # This will allow interactive conversations in the notebook, with JSON validation display
    thread = ChatHistoryAgentThread(chat_history=history_reducer)  # ChatHistoryAgentThread may not be explicitly typed, just track as object
    while True:
        user_input = input("Enter your request (or 'exit' to quit): ")
        if user_input.strip().lower() in {"exit", "quit"}:
            print("Exiting chat...")
            await runtime.stop_when_idle()
            break

        # Show user message
        html_output = "<div style='margin-bottom:10px'>"
        html_output += "<div style='font-weight:bold'>User:</div>"
        html_output += f"<div style='margin-left:20px'>{user_input}</div>"
        html_output += "</div>"

        # Get agent response using the existing chat function (mimics cell 12 'main')
        orchestration_result = await chat.invoke(
            task=user_input,
            runtime=runtime,
            thread=thread
        )
        value = await orchestration_result.get()
        agent_response = str(value)  # fallback to string if not structured
        
        try:
            # Try to format agent response as pretty JSON (like plan validation)
            html_output += "<div style='margin-bottom:20px'>"
            html_output += "<div style='font-weight:bold'>Modkit Subcommand Plan:</div>"
            html_output += f"<pre style='margin-left:20px; padding:10px; border-radius:5px;'>{agent_response}</pre>"
            html_output += "</div>"
        except Exception as e:
            html_output += "<div style='margin-bottom:20px; color:red;'>"
            html_output += "<div style='font-weight:bold'>Validation/Error:</div>"
            html_output += f"<pre style='margin-left:20px;'>{str(e)}</pre>"
            html_output += "</div>"
            html_output += "<div style='margin-bottom:20px;'>"
            html_output += "<div style='font-weight:bold'>Raw Agent Response:</div>"
            html_output += f"<div style='margin-left:20px; white-space:pre-wrap'>{agent_response}</div>"
            html_output += "</div>"

        html_output += "<hr>"
        display(HTML(html_output))

await interactive_chat()